# Notebook 03: Embedding Generation

Generates all embedding tensors and saves to Drive. Sequence-level embeddings may already exist (verify shapes before regenerating). Residue-level embeddings must be generated fresh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys

REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
REPO_DIR = '/content/antibody-property-prediction'
BRANCH = 'implementation'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project/Antibody_Project')
# DATA_DIR is in the repo (data/ at repo root) -- comes from src.config
EMBEDDING_DIR = DRIVE_ROOT / 'embeddings'
RESULTS_DIR = DRIVE_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set.")

In [ ]:
!apt-get install -y hmmer
!pip install -q fair-esm ablang2 anarci wandb

In [ ]:
!pip install -q --upgrade ipython

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
subprocess.run(['find', '/content/antibody-property-prediction', '-type', 'd',
                '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
               capture_output=True)
print("Autoreload enabled, pycache cleared.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import wandb
wandb.login()

## Imports

In [ ]:
from src.config import DEVICE, EMBEDDING_DIR, DATA_DIR
from src.data.abagym import load_abagym_antibody, load_abagym_sequences, get_all_mutation_site_indices
from src.data.sabdab import load_sabdab
from src.embeddings import esm2, ablang2
from src.embeddings.delta import compute_delta_sequence, compute_delta_residue

## Load Data

## ESM-2: Sequence-Level Embeddings (AbAgym)

If esm2_abagym.pt and esm2_abagym_wildtype.pt exist on Drive with correct shapes (5318, 2560) and (5, 2560), skip regeneration. Otherwise re-run.

## ESM-2: Sequence-Level Embeddings (SAbDab)

If esm2_sabdab.pt exists with shape (491, 2560), skip.

## ESM-2: Residue-Level Embeddings (AbAgym)

Generate esm2_abagym_residue_mutsite.pt and esm2_abagym_residue_wtsite.pt. Expected shapes: (5318, 1280) each.

## ESM-2: Delta Embeddings

Compute sequence-level deltas. If esm2_abagym_delta.pt exists with correct shape, verify by spot check rather than regenerating.

## AbLang2: Sequence-Level Embeddings (AbAgym)

If ablang2_abagym.pt and ablang2_abagym_wildtype.pt exist with shapes (5318, 960) and (5, 960), skip.

## AbLang2: Sequence-Level Embeddings (SAbDab)

## AbLang2: Residue-Level Embeddings (AbAgym)

Generate ablang2_abagym_residue_mutsite.pt and ablang2_abagym_residue_wtsite.pt. Expected shapes: (5318, 480) each.

## AbLang2: Delta Embeddings

## Verification

For all saved tensors: check shapes, assert no NaN or Inf, spot check a known mutation against expected amino acid.